# 04 Build Patient Features

This notebook builds AI-ready patient feature tables from normalized vital signs. It keeps feature engineering separate from the normalized relational import layer.

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DATA_DIR = Path('data/processed')
FEATURES_DATA_DIR = Path('data/features')

PATIENT_FILE = PROCESSED_DATA_DIR / 'patient.csv'
VITAL_SIGNS_FILE = PROCESSED_DATA_DIR / 'vital_signs.csv'

FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
patient_df = pd.read_csv(PATIENT_FILE)
vital_signs_df = pd.read_csv(VITAL_SIGNS_FILE, parse_dates=['measured_at'])

print('Patient shape:', patient_df.shape)
print('Vital signs shape:', vital_signs_df.shape)

Patient shape: (333, 13)
Vital signs shape: (30390, 10)


In [3]:
latest_vitals = (
    vital_signs_df
    .sort_values(['patient_id', 'vital_type', 'measured_at'])
    .groupby(['patient_id', 'vital_type'], as_index=False)
    .tail(1)
)

latest_vitals.head()

,id,patient_id,vital_type,value,unit,measured_at,source_patient_id,source_encounter_id,observation_code,description
183,5eab518d-e0b9-4142-bf87-3988706a731f,0399ec89-fa2f-4b0c-b844-f684c88e7e13,BLOOD_PRESSURE_DIASTOLIC,68.0,mmHg,2025-07-26 16:06:27+00:00,715724a5-2018-a4ef-8f62-42ec13f9c91d,715724a5-2018-a4ef-68f1-8b84bba638f3,8462-4,Diastolic Blood Pressure
184,beefb856-c7f0-4061-995c-f7d8865bc60e,0399ec89-fa2f-4b0c-b844-f684c88e7e13,BLOOD_PRESSURE_SYSTOLIC,122.0,mmHg,2025-07-26 16:06:27+00:00,715724a5-2018-a4ef-8f62-42ec13f9c91d,715724a5-2018-a4ef-68f1-8b84bba638f3,8480-6,Systolic Blood Pressure
185,2fbf856c-f9ff-4524-ba66-fd4d075bae57,0399ec89-fa2f-4b0c-b844-f684c88e7e13,BMI,34.6,kg/m2,2025-07-26 16:06:27+00:00,715724a5-2018-a4ef-8f62-42ec13f9c91d,715724a5-2018-a4ef-68f1-8b84bba638f3,39156-5,Body mass index (BMI) [Ratio]
191,0a8becbb-b0e4-4d4d-938c-8b9db3093868,0399ec89-fa2f-4b0c-b844-f684c88e7e13,CHOLESTEROL,166.1,mg/dL,2025-08-02 16:06:27+00:00,715724a5-2018-a4ef-8f62-42ec13f9c91d,715724a5-2018-a4ef-0fb8-a23d9186b99a,2093-3,Cholesterol [Mass/volume] in Serum or Plasma
192,7f5b6003-af0e-455b-beb8-94dc2fd66841,0399ec89-fa2f-4b0c-b844-f684c88e7e13,GLUCOSE,95.5,mg/dL,2025-08-02 16:06:27+00:00,715724a5-2018-a4ef-8f62-42ec13f9c91d,715724a5-2018-a4ef-0fb8-a23d9186b99a,2339-0,Glucose [Mass/volume] in Blood


In [4]:
latest_vitals_pivot = latest_vitals.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='first'
).reset_index()

latest_vitals_pivot.columns.name = None
latest_vitals_pivot.head()

,patient_id,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT
0,0399ec89-fa2f-4b0c-b844-f684c88e7e13,68.0,122.0,34.6,166.1,95.5,82.0,162.4,77.2,37.1,91.3
1,0444d74c-c96c-4b7d-a6b4-2d428a1616c6,63.0,85.0,27.3,215.8,118.0,62.0,165.0,NaN,NaN,74.2
2,0506591e-7b74-493a-861b-049ab9f721ac,61.0,111.0,NaN,NaN,NaN,88.0,70.7,NaN,NaN,8.7
3,0644623d-1798-4058-a8db-601134b91c7e,58.0,98.0,27.6,140.3,70.5,91.0,164.1,NaN,NaN,74.4
4,07cabd83-8663-45d5-b737-c13a41c58575,106.0,128.0,30.5,101.8,90.8,62.0,163.0,83.6,38.0,81.0


In [5]:
recent_30d = vital_signs_df.copy()
max_ts = recent_30d['measured_at'].max()
cutoff_ts = max_ts - pd.Timedelta(days=30)
recent_30d = recent_30d[recent_30d['measured_at'] >= cutoff_ts].copy()

mean_30d = recent_30d.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='mean'
).reset_index()

mean_30d = mean_30d.add_prefix('avg_30d_')
mean_30d = mean_30d.rename(columns={'avg_30d_patient_id': 'patient_id'})
mean_30d.head()

vital_type,patient_id,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT
0,10652a71-020f-4471-8d81-400972189cc7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.7,NaN
1,1a932bdf-fa0a-432e-8f64-d96d2ddeeaef,71.0,126.0,17.4,NaN,NaN,100.0,170.9,NaN,NaN,50.8
2,2b49b2ba-d1af-4f11-8b49-97d820351418,73.0,137.0,29.3,NaN,NaN,90.0,160.8,NaN,NaN,75.8
3,35449851-fb5d-46fe-8b63-02d3d7dc23b7,84.0,125.0,27.7,218.7,105.6,84.0,185.2,NaN,NaN,95.0
4,3af6b8f8-fcfb-4b04-b1c4-dcce6cea4ce1,84.0,106.0,27.4,NaN,91.4,76.0,160.8,NaN,NaN,70.8


In [6]:
weight_rows = vital_signs_df[vital_signs_df['vital_type'] == 'WEIGHT'].copy()
weight_rows = weight_rows.sort_values(['patient_id', 'measured_at'])

weight_trend = weight_rows.groupby('patient_id').agg(
    first_weight=('value', 'first'),
    latest_weight=('value', 'last'),
    first_weight_time=('measured_at', 'first'),
    latest_weight_time=('measured_at', 'last')
).reset_index()

weight_trend['weight_change'] = weight_trend['latest_weight'] - weight_trend['first_weight']
weight_trend.head()

,patient_id,first_weight,latest_weight,first_weight_time,latest_weight_time,weight_change
0,0399ec89-fa2f-4b0c-b844-f684c88e7e13,91.3,91.3,2016-06-04 16:06:27+00:00,2025-07-26 16:06:27+00:00,0.0
1,0444d74c-c96c-4b7d-a6b4-2d428a1616c6,74.2,74.2,2017-03-09 12:33:15+00:00,2025-04-24 12:33:15+00:00,0.0
2,0506591e-7b74-493a-861b-049ab9f721ac,4.0,8.7,2025-06-05 18:12:37+00:00,2026-02-12 18:12:37+00:00,4.7
3,0644623d-1798-4058-a8db-601134b91c7e,81.1,74.4,2005-06-19 13:22:36+00:00,2014-08-17 13:22:36+00:00,-6.7
4,07cabd83-8663-45d5-b737-c13a41c58575,81.0,81.0,2016-07-19 19:27:29+00:00,2026-02-24 19:27:29+00:00,0.0


In [7]:
patient_features_df = patient_df[['id', 'patient_number', 'birth_date', 'gender']].copy()
patient_features_df = patient_features_df.rename(columns={'id': 'patient_id'})

patient_features_df = patient_features_df.merge(latest_vitals_pivot, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(mean_30d, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(weight_trend[['patient_id', 'weight_change', 'latest_weight_time']], on='patient_id', how='left')

patient_features_df.head()

,patient_id,patient_number,birth_date,gender,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT,weight_change,latest_weight_time
0,ae453fcf-7b23-4d71-895b-02b3b9869996,P00000001,2019-09-10,MALE,78.0,113.0,15.8,NaN,NaN,61.0,111.2,NaN,37.8,19.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.4,2025-08-26 08:06:56+00:00
1,82dbd84d-0e88-4842-9772-87fafb05b3de,P00000002,2009-08-17,MALE,88.0,128.0,18.9,NaN,NaN,87.0,170.7,NaN,37.5,55.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.8,2025-09-29 02:57:01+00:00
2,6383e402-5279-484b-8911-6f865e07ac9d,P00000003,1986-06-01,FEMALE,77.0,132.0,27.6,135.3,NaN,90.0,158.3,NaN,37.2,69.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2023-08-19 22:33:58+00:00
3,8060a298-1110-4721-9cb0-98ae34ae9b70,P00000004,1981-01-17,FEMALE,81.0,97.0,28.2,192.0,82.9,68.0,164.8,NaN,NaN,76.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.7,2025-02-01 19:16:29+00:00
4,b2df7c38-d183-4ece-8e2c-5bfe007bd0c6,P00000005,1996-01-06,FEMALE,83.0,123.0,23.1,NaN,NaN,93.0,153.2,NaN,37.4,54.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.2,2025-03-22 05:51:11+00:00


In [8]:
feature_output_file = FEATURES_DATA_DIR / 'patient_features.csv'
patient_features_df.to_csv(feature_output_file, index=False)

print('Exported:', feature_output_file)

Exported: data\features\patient_features.csv


In [9]:
display(patient_features_df.head(20))

,patient_id,patient_number,birth_date,gender,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT,weight_change,latest_weight_time
0,ae453fcf-7b23-4d71-895b-02b3b9869996,P00000001,2019-09-10,MALE,78.0,113.0,15.8,NaN,NaN,61.0,111.2,NaN,37.8,19.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.4,2025-08-26 08:06:56+00:00
1,82dbd84d-0e88-4842-9772-87fafb05b3de,P00000002,2009-08-17,MALE,88.0,128.0,18.9,NaN,NaN,87.0,170.7,NaN,37.5,55.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.8,2025-09-29 02:57:01+00:00
2,6383e402-5279-484b-8911-6f865e07ac9d,P00000003,1986-06-01,FEMALE,77.0,132.0,27.6,135.3,NaN,90.0,158.3,NaN,37.2,69.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2023-08-19 22:33:58+00:00
3,8060a298-1110-4721-9cb0-98ae34ae9b70,P00000004,1981-01-17,FEMALE,81.0,97.0,28.2,192.0,82.9,68.0,164.8,NaN,NaN,76.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.7,2025-02-01 19:16:29+00:00
4,b2df7c38-d183-4ece-8e2c-5bfe007bd0c6,P00000005,1996-01-06,FEMALE,83.0,123.0,23.1,NaN,NaN,93.0,153.2,NaN,37.4,54.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.2,2025-03-22 05:51:11+00:00
5,4b7cabf2-aeac-498f-9c76-bab4e953015c,P00000006,2017-10-06,MALE,91.0,120.0,15.8,NaN,NaN,79.0,128.5,NaN,37.6,26.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.2,2025-10-03 01:33:26+00:00
6,aef1df15-7ac6-4ed1-b131-a0cfecacea36,P00000007,2007-04-03,FEMALE,92.0,115.0,22.6,NaN,NaN,67.0,163.1,NaN,NaN,60.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.2,2025-05-27 03:19:07+00:00
7,1b417512-6728-4150-8df7-38272c83043d,P00000008,1975-05-11,FEMALE,76.0,120.0,30.1,252.2,NaN,97.0,152.6,95.0,37.7,70.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.2,2025-05-11 16:22:47+00:00
8,a4fd5f68-d435-4581-a626-1874a81622df,P00000009,2023-04-03,FEMALE,81.0,129.0,16.9,NaN,NaN,71.0,96.4,NaN,NaN,15.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.4,2026-03-09 21:56:11+00:00
9,f5408bbf-3e6c-456a-b337-0cfe3b9176b8,P00000010,2010-07-12,MALE,93.0,117.0,16.6,NaN,NaN,96.0,175.2,NaN,NaN,50.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.0,2025-08-18 05:59:38+00:00
